# บทที่ 7: Recurrent Neural Networks (RNN, LSTM, GRU)

ใน Notebook นี้ เราจะ implement RNN, LSTM และ GRU จาก scratch พร้อมทั้งศึกษาปัญหา Vanishing Gradient

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

np.random.seed(42)

## 2. Simple RNN Implementation

In [ ]:
class SimpleRNN:
    """
    Simple RNN implementation from scratch
    
    h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    y_t = softmax(W_hy @ h_t + b_y)
    """
    
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        
        # Initialize weights
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))
        
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.b_y = np.zeros((output_size, 1))
        
    def forward(self, x_sequence):
        """
        Forward pass through sequence
        
        Parameters:
        - x_sequence: list of inputs [x_1, x_2, ..., x_T]
        """
        h = np.zeros((self.hidden_size, 1))
        self.hidden_states = [h]
        self.inputs = []
        
        outputs = []
        
        for x in x_sequence:
            x = x.reshape(-1, 1)
            self.inputs.append(x)
            
            # Hidden state
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            self.hidden_states.append(h)
            
            # Output
            y = self.W_hy @ h + self.b_y
            outputs.append(y)
            
        return outputs
    
    def predict(self, x_sequence):
        """Get final output"""
        outputs = self.forward(x_sequence)
        return outputs[-1]

## 3. ทดสอบ Simple RNN

In [ ]:
# Create RNN
rnn = SimpleRNN(input_size=2, hidden_size=4, output_size=1)

# Test sequence
sequence = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1])
]

outputs = rnn.forward(sequence)

print("=== RNN Forward Pass ===")
for t, (x, h, y) in enumerate(zip(sequence, rnn.hidden_states[1:], outputs)):
    print(f"\nTime step {t+1}:")
    print(f"  Input: {x}")
    print(f"  Hidden state: {h.flatten()}")
    print(f"  Output: {y.flatten()}")

## 4. Vanishing Gradient Demonstration

In [ ]:
def demonstrate_vanishing_gradient():
    """
    สาธิตปัญหา Vanishing Gradient
    
    ใน RNN, gradient ไหลย้อนกลับผ่าน time steps หลายๆ ครั้ง
    ทำให้ gradient หดตัวหรือขยายตัว
    """
    # tanh derivative: 1 - tanh²(x)
    # Maximum value of tanh derivative is 1 (at x=0)
    # But typically < 1
    
    time_steps = 20
    
    # Simulate gradient flow with tanh
    gradients_tanh = [1.0]
    for t in range(time_steps):
        # Assume typical tanh derivative ~ 0.5
        grad = gradients_tanh[-1] * 0.5
        gradients_tanh.append(grad)
        
    # Simulate gradient flow with ReLU (for comparison)
    gradients_relu = [1.0]
    for t in range(time_steps):
        # ReLU derivative is 0 or 1
        grad = gradients_relu[-1] * 1.0  # When active
        gradients_relu.append(grad)
        
    plt.figure(figsize=(10, 6))
    plt.semilogy(gradients_tanh, 'r-o', label='Tanh (Vanishing)')
    plt.semilogy(gradients_relu, 'b-s', label='ReLU (No Vanishing)')
    plt.xlabel('Time Steps')
    plt.ylabel('Gradient Magnitude (log scale)')
    plt.title('Vanishing Gradient Problem in RNN')
    plt.legend()
    plt.show()
    
    print(f"\nTanh gradient after 20 steps: {gradients_tanh[-1]:.6f}")
    print(f"ReLU gradient after 20 steps: {gradients_relu[-1]:.6f}")

demonstrate_vanishing_gradient()

## 5. LSTM Cell Implementation

In [ ]:
class LSTMCell:
    """
    LSTM Cell with 3 Gates:
    - Forget Gate: f_t = σ(W_f @ [h_{t-1}, x_t] + b_f)
    - Input Gate: i_t = σ(W_i @ [h_{t-1}, x_t] + b_i)
    - Output Gate: o_t = σ(W_o @ [h_{t-1}, x_t] + b_o)
    
    Cell State Update:
    - C̃_t = tanh(W_c @ [h_{t-1}, x_t] + b_c)
    - C_t = f_t * C_{t-1} + i_t * C̃_t
    - h_t = o_t * tanh(C_t)
    """
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        
        # Forget gate
        self.W_f = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_f = np.zeros((hidden_size, 1))
        
        # Input gate
        self.W_i = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_i = np.zeros((hidden_size, 1))
        
        # Output gate
        self.W_o = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_o = np.zeros((hidden_size, 1))
        
        # Cell state (candidate)
        self.W_c = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_c = np.zeros((hidden_size, 1))
        
    def forward(self, x, h_prev, c_prev):
        """Single forward step"""
        # Concatenate h_prev and x
        combined = np.vstack([h_prev, x.reshape(-1, 1)])
        
        # Gates
        f_t = self._sigmoid(self.W_f @ combined + self.b_f)  # Forget gate
        i_t = self._sigmoid(self.W_i @ combined + self.b_i)  # Input gate
        o_t = self._sigmoid(self.W_o @ combined + self.b_o)  # Output gate
        
        # Cell state candidate
        c_tilde = np.tanh(self.W_c @ combined + self.b_c)
        
        # Update cell state
        c_t = f_t * c_prev + i_t * c_tilde
        
        # Update hidden state
        h_t = o_t * np.tanh(c_t)
        
        return h_t, c_t, (f_t, i_t, o_t, c_tilde)
    
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


class LSTM:
    """Full LSTM network"""
    
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)
        self.W_out = np.random.randn(output_size, hidden_size) * 0.01
        self.b_out = np.zeros((output_size, 1))
        
    def forward(self, x_sequence):
        """Forward pass through sequence"""
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        
        self.gates_history = []
        
        for x in x_sequence:
            h, c, gates = self.cell.forward(x, h, c)
            self.gates_history.append(gates)
            
        # Final output
        y = self.W_out @ h + self.b_out
        return y, h

## 6. ทดสอบ LSTM

In [ ]:
# Create LSTM
lstm = LSTM(input_size=2, hidden_size=4, output_size=1)

# Test sequence
sequence = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1])
]

output, h_final = lstm.forward(sequence)

print("=== LSTM Forward Pass ===")
print(f"\nFinal hidden state: {h_final.flatten()}")
print(f"Output: {output.flatten()}")

print("\n=== Gates at each time step ===")
for t, (f, i, o, c_tilde) in enumerate(lstm.gates_history):
    print(f"\nTime step {t+1}:")
    print(f"  Forget gate: {f.flatten()}")
    print(f"  Input gate: {i.flatten()}")
    print(f"  Output gate: {o.flatten()}")

## 7. GRU Cell Implementation

In [ ]:
class GRUCell:
    """
    GRU Cell with 2 Gates:
    - Reset Gate: r_t = σ(W_r @ [h_{t-1}, x_t])
    - Update Gate: z_t = σ(W_z @ [h_{t-1}, x_t])
    
    Hidden State Update:
    - h̃_t = tanh(W_h @ [r_t * h_{t-1}, x_t])
    - h_t = (1 - z_t) * h_{t-1} + z_t * h̃_t
    """
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        
        # Reset gate
        self.W_r = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        
        # Update gate
        self.W_z = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        
        # Candidate hidden state
        self.W_h = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        
    def forward(self, x, h_prev):
        """Single forward step"""
        combined = np.vstack([h_prev, x.reshape(-1, 1)])
        
        # Gates
        r_t = self._sigmoid(self.W_r @ combined)  # Reset gate
        z_t = self._sigmoid(self.W_z @ combined)  # Update gate
        
        # Candidate hidden state
        combined_reset = np.vstack([r_t * h_prev, x.reshape(-1, 1)])
        h_tilde = np.tanh(self.W_h @ combined_reset)
        
        # Update hidden state
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        
        return h_t, (r_t, z_t, h_tilde)
    
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


class GRU:
    """Full GRU network"""
    
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)
        self.W_out = np.random.randn(output_size, hidden_size) * 0.01
        self.b_out = np.zeros((output_size, 1))
        
    def forward(self, x_sequence):
        h = np.zeros((self.hidden_size, 1))
        self.gates_history = []
        
        for x in x_sequence:
            h, gates = self.cell.forward(x, h)
            self.gates_history.append(gates)
            
        y = self.W_out @ h + self.b_out
        return y, h

## 8. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: RNN Hidden State

In [ ]:
# ให้ RNN มีค่าดังนี้:
# W_xh = [[0.5]], W_hh = [[0.8]], b_h = [[0.1]]
# จงคำนวณ hidden state สำหรับ sequence x = [1, 2, 3]

W_xh = np.array([[0.5]])
W_hh = np.array([[0.8]])
b_h = np.array([[0.1]])

sequence = [1, 2, 3]
h = 0  # Initial hidden state

print("=== RNN Hidden State Calculation ===")
for t, x in enumerate(sequence):
    h = np.tanh(W_xh @ np.array([[x]]) + W_hh @ np.array([[h]]) + b_h)
    print(f"Time step {t+1}: x={x}, h={h[0,0]:.4f}")

### แบบฝึกหัดที่ 2: LSTM Gates

In [ ]:
# ให้ Forget gate output = 0.8, Input gate output = 0.3
# Previous cell state = 1.0, Candidate cell state = 0.5
# จงคำนวณ new cell state

f_t = 0.8  # Forget gate
i_t = 0.3  # Input gate
c_prev = 1.0  # Previous cell state
c_tilde = 0.5  # Candidate cell state

c_new = f_t * c_prev + i_t * c_tilde
print(f"New cell state: {c_new:.4f}")
print(f"Calculation: {f_t} × {c_prev} + {i_t} × {c_tilde} = {c_new:.4f}")

### แบบฝึกหัดที่ 3: GRU Update

In [ ]:
# ให้ Update gate = 0.7, Previous hidden = 0.5, Candidate hidden = 0.9
# จงคำนวณ new hidden state

z_t = 0.7  # Update gate
h_prev = 0.5  # Previous hidden state
h_tilde = 0.9  # Candidate hidden state

h_new = (1 - z_t) * h_prev + z_t * h_tilde
print(f"New hidden state: {h_new:.4f}")
print(f"Calculation: (1 - {z_t}) × {h_prev} + {z_t} × {h_tilde} = {h_new:.4f}")

### แบบฝึกหัดที่ 4: คำนวณ Parameters

In [ ]:
# จงคำนวณจำนวน parameters ของ LSTM ที่มี input_size=10, hidden_size=20

input_size = 10
hidden_size = 20

# LSTM has 4 sets of weights (forget, input, output, cell)
# Each set: W (hidden_size × (input_size + hidden_size)) + b (hidden_size)
params_per_gate = hidden_size * (input_size + hidden_size) + hidden_size
total_params = 4 * params_per_gate

print(f"LSTM Parameters:")
print(f"Input size: {input_size}")
print(f"Hidden size: {hidden_size}")
print(f"Parameters per gate: {params_per_gate}")
print(f"Total LSTM parameters: {total_params}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **Simple RNN**: โครงสร้างพื้นฐาน มีปัญหา Vanishing Gradient
2. **LSTM**: มี 3 Gates (Forget, Input, Output) แก้ปัญหา Long-term Dependencies
3. **GRU**: มี 2 Gates (Reset, Update) เร็วกว่า LSTM แต่มีประสิทธิภาพใกล้เคียง